# Breakout Room #3 — Identificar los 4 cuadrantes

**Objetivo:**
- Detectar a los clientes que están en los cuatro cuadrantes.
- ¿Lograremos ver en cuáles es necesario invertir?

Esta es la base del experimento real de retención de Starbucks Rewards: 14,000 clientes, la mitad
recibió una oferta de retención (`treatment`), la otra mitad no (`control`) — asignados al azar.
Donde dice **`# TODO`** hay algo para que ustedes completen.


In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/Volumes/workspace/default/semana3/uplift_experiment.csv")
print(f"{len(df):,} clientes · {(df['treatment_group']=='treatment').sum():,} tratados · {(df['treatment_group']=='control').sum():,} control")
df.head()


## Paso 1 — recuerden la fórmula de "uplift en una línea"

```
uplift(cliente) = P(compra | lo contacto) − P(compra | no lo contacto)
```

El problema: para UN cliente real, solo observamos una de las dos mitades (su contrafactual no
existe — ya lo vimos). La solución práctica: un **T-learner**. Entrenamos DOS modelos separados:

- Un modelo que solo vio clientes **tratados**, para estimar P(compra | lo contacto).
- Un modelo que solo vio clientes de **control**, para estimar P(compra | no lo contacto).

Después, le preguntamos a LOS DOS modelos sobre TODOS los clientes (tratados y de control por
igual) — así cada cliente termina con sus dos probabilidades, aunque en la realidad solo vivió una.


In [0]:
FEATURES = ["recency_days", "frequency", "monetary", "email_open_rate"]

tratados = df[df["treatment_group"] == "treatment"]
control = df[df["treatment_group"] == "control"]

print(f"Entrenamiento del modelo de tratados: {len(tratados):,} filas")
print(f"Entrenamiento del modelo de control:  {len(control):,} filas")


In [0]:
control.head()

## Paso 2 — pregúntenle a los DOS modelos sobre TODOS los clientes

Aquí está el truco del T-learner: no importa si un cliente de verdad fue tratado o no, le pedimos
a `modelo_tratado` su probabilidad COMO SI lo hubieran tratado, y a `modelo_control` su
probabilidad COMO SI no lo hubieran tratado.


## Paso 3 — clasifiquen a cada cliente en uno de los 4 cuadrantes

Usamos la mediana de cada probabilidad como punto de corte entre "alta" y "baja".


In [0]:
# --- Train T-learner models (completes TODOs from cells 6 & 8) ---
FEATURES = ["recency_days", "frequency", "monetary", "email_open_rate"]

tratados = df[df["treatment_group"] == "treatment"]
control = df[df["treatment_group"] == "control"]

modelo_tratado = LogisticRegression()
modelo_tratado.fit(tratados[FEATURES], tratados["responded_90d"])

modelo_control = LogisticRegression()
modelo_control.fit(control[FEATURES], control["responded_90d"])

X_todos = df[FEATURES]
df["p_tratado"] = modelo_tratado.predict_proba(X_todos)[:, 1]
df["p_control"] = modelo_control.predict_proba(X_todos)[:, 1]
df["uplift_estimado"] = df["p_tratado"] - df["p_control"]

# --- Paso 3: clasificar en cuadrantes ---
corte_tratado = df["p_tratado"].median()
corte_control = df["p_control"].median()

def cuadrante(row):
    alto_t = row["p_tratado"] >= corte_tratado
    alto_c = row["p_control"] >= corte_control
    if alto_t and alto_c:
        return "Sure Thing"
    if alto_t and not alto_c:
        return "Persuadable"
    if not alto_t and alto_c:
        return "Sleeping Dog"
    return "Lost Cause"

df["cuadrante"] = df.apply(cuadrante, axis=1)
df["cuadrante"].value_counts(normalize=True).rename("% de la base").mul(100).round(1)


## Paso 4 — ¿en cuáles es necesario invertir?

Comparen tres reglas de targeting, igual que en la slide de políticas: contactar al 30% con mayor
RIESGO de no responder (estilo churn) vs. contactar al 30% con mayor UPLIFT ESTIMADO vs. un 30% al
azar. Midan el valor real usando `value_90d`, comparando tratados contra control DENTRO de cada
grupo seleccionado

In [0]:
df.head()

In [0]:
def valor_de_la_politica(seleccionados):
    sel = df.loc[seleccionados]
    valor_tratado = sel.loc[sel["treatment_group"] == "treatment", "value_90d"].mean()
    valor_control = sel.loc[sel["treatment_group"] == "control", "value_90d"].mean()
    return valor_tratado - valor_control, len(sel)

n_contactar = int(len(df) * 0.30)

# Regla 1: al azar
idx_azar = df.sample(n=n_contactar, random_state=42).index
uplift_azar, n_azar = valor_de_la_politica(idx_azar)

# Regla 2: por "riesgo" (al estilo churn) -- aquí lo aproximamos con p_control baja
# (baja probabilidad de comprar sin ayuda = "en riesgo")
idx_riesgo = df.sort_values("p_control").head(n_contactar).index
uplift_riesgo, n_riesgo = valor_de_la_politica(idx_riesgo)

# Regla 3: por uplift estimado (T-learner)
idx_uplift = df.sort_values("uplift_estimado", ascending=False).head(n_contactar).index
uplift_modelo, n_modelo = valor_de_la_politica(idx_uplift)

resultado = pd.DataFrame({
    "regla": ["Al azar", "Por riesgo (estilo churn)", "Por uplift (T-learner)"],
    "clientes_contactados": [n_azar, n_riesgo, n_modelo],
    "valor_incremental_por_cliente": [uplift_azar, uplift_riesgo, uplift_modelo],
})
resultado["valor_incremental_total"] = resultado["valor_incremental_por_cliente"] * resultado["clientes_contactados"]
resultado.round(2)


## Paso 5 — la gráfica: composición del 30% seleccionado por cada regla


In [0]:
import matplotlib.pyplot as plt

comp_riesgo = df.loc[idx_riesgo, "cuadrante"].value_counts(normalize=True).reindex(
    ["Lost Cause", "Sure Thing", "Sleeping Dog", "Persuadable"]).fillna(0) * 100
comp_uplift = df.loc[idx_uplift, "cuadrante"].value_counts(normalize=True).reindex(
    ["Lost Cause", "Sure Thing", "Sleeping Dog", "Persuadable"]).fillna(0) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
colors = {"Lost Cause": "#9A988E", "Sure Thing": "#C8C6BC", "Sleeping Dog": "#C4512F", "Persuadable": "#00704A"}

for ax, comp, titulo in zip(axes, [comp_riesgo, comp_uplift], ["Targeting por riesgo", "Targeting por uplift"]):
    ax.bar(comp.index, comp.values, color=[colors[c] for c in comp.index])
    ax.set_title(titulo)
    ax.set_ylabel("% del 30% contactado")
    for i, v in enumerate(comp.values):
        ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("/Volumes/workspace/default/semana3/composicion_cuadrantes.png", dpi=150)
plt.show()


In [0]:
# Comparación de "tocar" o "no tocar" a un sleeping dog.
sd = df[df["cuadrante"] == "Sleeping Dog"]
sd_t = sd.loc[sd["treatment_group"] == "treatment", "value_90d"].mean()
sd_c = sd.loc[sd["treatment_group"] == "control", "value_90d"].mean()
print(f"Sleeping Dogs: {len(sd):,} ({len(sd)/len(df):.1%} de la base)")
print(f"  valor si tratado=${sd_t:.2f}  valor si control=${sd_c:.2f}  efecto=${sd_t-sd_c:+.2f}")

## Para la Clase

**1. ¿Qué regla dejó más valor incremental total: por riesgo o por uplift?**

   La regla por uplift dejó el mayor valor incremental total, con $34,229.38, mientras que la estrategia por riesgo generó solamente $1,264.60

**2. Miren la composición del 30% que targetea "por riesgo" — ¿cuántos Sleeping Dogs se colaron ahí?
   Eso es exactamente el costo de confundir "en riesgo" con "vale la pena contactar".**

   Se colaron 0 Sleeping Dogs en el targeting por riesgo. Sin embargo, el problema de la estrategia sigue existiendo: seleccionó exclusivamente Lost Causes, clientes que parecen estar en riesgo, pero sobre quienes la intervención tiene muy poco impacto.

**3. Con su propia tabla del Paso 4 en la mano: **¿en qué cuadrante es necesario invertir, y en
   cuál definitivamente NO?****

   En los persuadibles, es un cliente que probablemente no compraría por sí solo, pero sí puede hacerlo si recibe la oferta.
